In [1]:
!pip install transformers datasets sentencepiece accelerate evaluate sacrebleu -q

In [2]:
!pip uninstall -y torch torchaudio torchvision
!pip install torch torchaudio --no-cache-dir -q

Found existing installation: torch 2.12.1
Uninstalling torch-2.12.1:
  Successfully uninstalled torch-2.12.1
Found existing installation: torchaudio 2.11.0
Uninstalling torchaudio-2.11.0:
  Successfully uninstalled torchaudio-2.11.0


In [3]:
!pip install "transformers==4.44.2" -q

In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
import torch

model_name = "Helsinki-NLP/opus-mt-ar-en"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

dataset = load_dataset(
    "csv",
    data_files={"train": "/home/jovyan/projects/translate/final_ar_en_dataset.csv"}
)

dataset = dataset["train"].train_test_split(test_size=0.1)
train_ds = dataset["train"]
val_ds   = dataset["test"]

max_length = 128

def preprocess(batch):
    model_inputs = tokenizer(
        batch["ar"], truncation=True, max_length=max_length
    )
    labels = tokenizer(
        batch["en"], truncation=True, max_length=max_length
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(
    preprocess,
    batched=True,
    remove_columns=dataset["train"].column_names,
    num_proc=4
)

train_ds = tokenized_dataset["train"]
val_ds   = tokenized_dataset["test"]

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

training_args = TrainingArguments(
    output_dir="./opus-ar-en-custom",
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    learning_rate=3e-4,
    num_train_epochs=2,
    save_steps=2000,
    eval_steps=2000,
    logging_steps=200,
    bf16=True,
    tf32=True,
    group_by_length=True,
    dataloader_num_workers=4,
    optim="adamw_torch_fused",
    save_total_limit=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator
)

trainer.train()

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/opt/conda/lib/python3.12/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
/opt/conda/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map (num_proc=4):   0%|          | 0/1135915 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/126213 [00:00<?, ? examples/s]

Step,Training Loss
200,2.140200
400,1.770800
600,1.713500
800,1.666400
1000,1.610500
1200,1.589700
1400,1.569800
1600,1.545800
1800,1.541200
2000,1.519700


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[62833]], 'forced_eos_token_id': 0}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[62833]], 'forced_eos_token_id': 0}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strate

TrainOutput(global_step=35498, training_loss=1.221205495849073, metrics={'train_runtime': 3916.8196, 'train_samples_per_second': 580.019, 'train_steps_per_second': 9.063, 'total_flos': 1.0689968553984e+16, 'train_loss': 1.221205495849073, 'epoch': 2.0})

In [9]:
save_path = "/home/jovyan/projects/translate/opus-ar-en-custom/final-model"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print("saved to:", save_path)

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[62833]], 'forced_eos_token_id': 0}


saved to: /home/jovyan/projects/translate/opus-ar-en-custom/final-model


In [11]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

model_path = "/home/jovyan/projects/translate/opus-ar-en-custom/final-model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

translator = pipeline("translation", model=model, tokenizer=tokenizer)

texts = [
    "اريد تجربت صنع طائرة شراعية لكن الامر يبدو صعبا نوعا ما",
    "أنا أدرس الذكاء الاصطناعي",
    "هذا نموذج ترجمة عربي إلى إنجليزي"
]

for text in texts:
    out = translator(text, max_length=128)
    print("AR:", text)
    print("EN:", out[0]["translation_text"])
    print("-" * 40)

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


AR: اريد تجربت صنع طائرة شراعية لكن الامر يبدو صعبا نوعا ما
EN: I want to try and make a drone, but it's kind of hard.
----------------------------------------
AR: أنا أدرس الذكاء الاصطناعي
EN: I'm studying artificial intelligence.
----------------------------------------
AR: هذا نموذج ترجمة عربي إلى إنجليزي
EN: This is an Arab translation model for English .
----------------------------------------


In [12]:
import shutil

folder_path = "/home/jovyan/projects/translate/opus-ar-en-custom/final-model"
zip_name = "/home/jovyan/projects/translate/final-model"

shutil.make_archive(zip_name, "zip", folder_path)
print("ZIP saved")

ZIP saved
